<a href="https://colab.research.google.com/github/mipaillafil/telco-ml-e1/blob/main/machine_telco_procesado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Telco Customer Churn — Preparación de datos**

En este notebook se prepara el dataset limpio para utilizarlo en algoritmos de Machine Learning.

##**Importar Librerias**

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

##**Cargar Dataset Limpio**


In [12]:
df= pd.read_csv('https://raw.githubusercontent.com/mipaillafil/telco-ml-e1/main/data/processed/Telco_Customer_Churn_Clean.csv')
print("Dimensiones:", df.shape)
display(df.head())

Dimensiones: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


##**Variable Objetivo**

In [13]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("Distribución de Churn:")
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).mul(100).round(2))


Distribución de Churn:


,count
Churn,
0,5174
1,1869


,proportion
Churn,
0,73.46
1,26.54


##**Separación de Variables**

In [14]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

# customerID ya fue eliminado en el notebook de limpieza.
if "customerID" in X.columns:
    X = X.drop(columns=["customerID"])

print("Variables predictoras:", X.shape)
print("Variable objetivo:", y.shape)

Variables predictoras: (7043, 19)
Variable objetivo: (7043,)


##**Codificación de Variables Categoricas**

In [15]:
# Convertimos las variables categóricas a variables numéricas.
X = pd.get_dummies(X, drop_first=True)

print("Cantidad de variables después de One-Hot Encoding:", X.shape[1])
display(X.head())

Cantidad de variables después de One-Hot Encoding: 30


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False


##**División Train/Test**
Se utiliza 80% para entrenamiento y 20% para prueba.
`stratify=y` mantiene una proporción similar de Churn en ambos conjuntos.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nProporción Churn en Train:")
display(y_train.value_counts(normalize=True).mul(100).round(2))

X_train: (5634, 30)
X_test: (1409, 30)
y_train: (5634,)
y_test: (1409,)

Proporción Churn en Train:


,proportion
Churn,
0,73.46
1,26.54


In [18]:
print("\nProporción Churn en Test:")
display(y_test.value_counts(normalize=True).mul(100).round(2))


Proporción Churn en Test:


,proportion
Churn,
0,73.46
1,26.54


##**Escalamiento**
El `StandardScaler` se ajusta solamente con `X_train`.
Después se utiliza esa misma transformación sobre `X_test`.

In [19]:
columnas_numericas = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[columnas_numericas] = scaler.fit_transform(X_train[columnas_numericas])
X_test[columnas_numericas] = scaler.transform(X_test[columnas_numericas])
print("Escalamiento realizado sin utilizar información del conjunto de prueba.")


Escalamiento realizado sin utilizar información del conjunto de prueba.


##**Validación de Calidad**

In [23]:
print("Nulos en X_train:", X_train.isnull().sum().sum())
print("Nulos en X_test:", X_test.isnull().sum().sum())

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

# Validar que todas las variables sean numéricas y finitas
X_train_numeric = X_train.astype(float).to_numpy()
X_test_numeric = X_test.astype(float).to_numpy()

assert np.isfinite(X_train_numeric).all()
assert np.isfinite(X_test_numeric).all()

print("Validaciones completadas correctamente.")

Nulos en X_train: 0
Nulos en X_test: 0
Validaciones completadas correctamente.


In [25]:
import os

# Crear el directorio si no existe
os.makedirs('data/processed', exist_ok=True)

# Guardar los conjuntos de datos
X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print("Conjuntos X_train, X_test, y_train, y_test guardados en 'data/processed/'.")

Conjuntos X_train, X_test, y_train, y_test guardados en 'data/processed/'.


##**Conclusión**
Los datos fueron transformados a un formato adecuado para Machine Learning.
Se realizó One-Hot Encoding, división Train/Test y escalamiento evitando data leakage.
